Import libraries

In [1]:
import os
import re
import pandas as pd
import numpy as np
from statsmodels.sandbox.regression.gmm import GMM

In [2]:
import stata_setup
stata_setup.config(os.path.join('/', 'Applications', 'Stata'), 'se')
from pystata import stata


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      18.0
___/   /   /___/   /   /___/       SE—Standard Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-STATA-PC        https://www.stata.com
                                   979-696-4600        stata@stata.com

Stata license: Unlimited-user network, expiring 13 Dec 2025
Serial number: 401809347100
  Licensed to: Diana Tagliaferri
               

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Set directory

In [3]:
cd = os.getcwd()
print(cd)

/Users/dianatagliaferri/Library/CloudStorage/OneDrive-LondonBusinessSchool/Documents/Econometrics I/ps4


## Problem 2

**Problem 2a**

Read the dataset

In [4]:
ccapm = pd.read_excel('ccapm.xlsx', names = ['cratio', 'rrate', 'e'], header=None)

# Add columns for the lagged variables
for col in ccapm.columns: ccapm[f'{col}_lag'] = ccapm[col].shift(1)

# Drop the first row with NaN values due to lagging
ccapm = ccapm.dropna().reset_index(drop=True)

# Stack all variables in one array
data = ccapm[['cratio', 'rrate', 'cratio_lag', 'rrate_lag', 'e_lag']].to_numpy()

ccapm.head()

,cratio,rrate,e,cratio_lag,rrate_lag,e_lag
0,0.991563,1.035976,0.106118,1.010612,1.006276,0.034638
1,1.009609,1.018062,-0.084368,0.991563,1.035976,0.106118
2,0.999425,0.996259,0.096085,1.009609,1.018062,-0.084368
3,0.996430,1.034679,-0.005752,0.999425,0.996259,0.096085
4,1.002306,0.986218,-0.035698,0.996430,1.034679,-0.005752


Specify moment conditions

In [5]:
class GMM_ccapm(GMM):

    def __init__(self, *args, **kwds):
        # Set appropriate counts for moment conditions and parameters
        kwds.setdefault('k_moms', 4)
        kwds.setdefault('k_params', 2)
        super(GMM_ccapm, self).__init__(*args, **kwds)

    def momcond(self, params):
        beta, gamma = params
        cratio, rrate = self.endog.T
        cratio_lag, rrate_lag, e_lag = self.exog.T
        error1 = beta * cratio**(-gamma) * rrate -1
        error2 = (beta * cratio**(-gamma) * rrate -1) * cratio_lag
        error3 = (beta * cratio**(-gamma) * rrate -1) * rrate_lag
        error4 = (beta * cratio**(-gamma) * rrate -1) * e_lag
        g = np.column_stack((error1, error2, error3, error4))
        return g     

Perform estimation

In [6]:
model = GMM_ccapm(endog=data[:, 0:2], exog=data[:, 2:], instrument=None)
params0 = np.array([1, 1])
res = model.fit(params0, maxiter=2)
print(res.summary())

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 1
         Function evaluations: 3
         Gradient evaluations: 3
Optimization terminated successfully.
         Current function value: 0.005432
         Iterations: 5
         Function evaluations: 10
         Gradient evaluations: 10
                              GMM_ccapm Results                               
Dep. Variable:           ['y1', 'y2']   Hansen J:                        1.287
Model:                      GMM_ccapm   Prob (Hansen J):                 0.525
Method:                           GMM                                         
Date:                Mon, 01 Dec 2025                                         
Time:                        22:54:40                                         
No. Observations:                 237                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------

Export as LaTeX table

In [7]:
for table in res.summary().tables:
    print(table.as_latex_tabular())

\begin{center}
\begin{tabular}{lclc}
\toprule
\textbf{Dep. Variable:}    &   ['y1', 'y2']   & \textbf{  Hansen J:          } &    1.287  \\
\textbf{Model:}            &    GMM\_ccapm    & \textbf{  Prob (Hansen J):   } &   0.525   \\
\textbf{Method:}           &       GMM        & \textbf{                     } &           \\
\textbf{Date:}             & Mon, 01 Dec 2025 & \textbf{                     } &           \\
\textbf{Time:}             &     22:54:40     & \textbf{                     } &           \\
\textbf{No. Observations:} &         237      & \textbf{                     } &           \\
\bottomrule
\end{tabular}
%\caption{GMM_ccapm Results}
\end{center}
\begin{center}
\begin{tabular}{lcccccc}
\toprule
            & \textbf{coef} & \textbf{std err} & \textbf{z} & \textbf{P$> |$z$|$} & \textbf{[0.025} & \textbf{0.975]}  \\
\midrule
\textbf{x2} &       0.9977  &        0.004     &   230.377  &         0.000        &        0.989    &        1.006     \\
\textbf{x3} &      

In [8]:
latex_str = res.summary().tables[1].as_latex_tabular()
replacements = {
    r'\textbf{coef}': 'Estimate',
    r'\textbf{std err}': 'Std. error',
    r'\textbf{z}': '$Z$-value',
    r'\textbf{P$> |$z$|$}': '$p$-value',
    r'\textbf{[0.025}': r'95\% CI Lower',
    r'\textbf{0.975]}': r'95\% CI Upper',
    r'\textbf{x2}': '$\\beta$',
    r'\textbf{x3}': '$\\gamma$',
}
for old, new in replacements.items(): latex_str = latex_str.replace(old, new)
latex_str = r'''
\begin{table}[!htbp]
\caption{GMM Estimates of the CCAPM}
\label{tab:problem2a}
''' + '\n'+ latex_str + '\n' + r'''
\end{table}
'''
with open('p2a_table.tex', 'w') as f:
    f.write(latex_str)

**Problem 2b**

Re-run estimation implementing Newey-West-type HAC with truncation lag equal to 5

In [9]:
model = GMM_ccapm(endog=data[:, 0:2], exog=data[:, 2:], instrument=None)
params0 = np.array([1, 1])
res = model.fit(params0, maxiter=2, weights_method='hac', wargs={'maxlag':5})
print(res.summary())

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 1
         Function evaluations: 3
         Gradient evaluations: 3
Optimization terminated successfully.
         Current function value: 0.008025
         Iterations: 5
         Function evaluations: 10
         Gradient evaluations: 10
                              GMM_ccapm Results                               
Dep. Variable:           ['y1', 'y2']   Hansen J:                        1.902
Model:                      GMM_ccapm   Prob (Hansen J):                 0.386
Method:                           GMM                                         
Date:                Mon, 01 Dec 2025                                         
Time:                        22:54:40                                         
No. Observations:                 237                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------

Export as LaTeX table

In [10]:
for table in res.summary().tables:
    print(table.as_latex_tabular())

\begin{center}
\begin{tabular}{lclc}
\toprule
\textbf{Dep. Variable:}    &   ['y1', 'y2']   & \textbf{  Hansen J:          } &    1.902  \\
\textbf{Model:}            &    GMM\_ccapm    & \textbf{  Prob (Hansen J):   } &   0.386   \\
\textbf{Method:}           &       GMM        & \textbf{                     } &           \\
\textbf{Date:}             & Mon, 01 Dec 2025 & \textbf{                     } &           \\
\textbf{Time:}             &     22:54:40     & \textbf{                     } &           \\
\textbf{No. Observations:} &         237      & \textbf{                     } &           \\
\bottomrule
\end{tabular}
%\caption{GMM_ccapm Results}
\end{center}
\begin{center}
\begin{tabular}{lcccccc}
\toprule
            & \textbf{coef} & \textbf{std err} & \textbf{z} & \textbf{P$> |$z$|$} & \textbf{[0.025} & \textbf{0.975]}  \\
\midrule
\textbf{x2} &       0.9979  &        0.004     &   226.569  &         0.000        &        0.989    &        1.006     \\
\textbf{x3} &      

In [11]:
latex_str = res.summary().tables[1].as_latex_tabular()
replacements = {
    r'\textbf{coef}': 'Estimate',
    r'\textbf{std err}': 'Std. error',
    r'\textbf{z}': '$Z$-value',
    r'\textbf{P$> |$z$|$}': '$p$-value',
    r'\textbf{[0.025}': r'95\% CI Lower',
    r'\textbf{0.975]}': r'95\% CI Upper',
    r'\textbf{x2}': '$\\beta$',
    r'\textbf{x3}': '$\\gamma$',
}
for old, new in replacements.items(): latex_str = latex_str.replace(old, new)
latex_str = r'''
\begin{table}[!htbp]
\caption{GMM Estimates of the CCAPM (HAC weighting matrix, truncation lag 5)}
\label{tab:problem2b}
''' + '\n'+ latex_str + '\n' + r'''
\end{table}
'''
with open('p2b_table.tex', 'w') as f:
    f.write(latex_str)

**Problem 2c**

Using the setting from Problem 2b, test the hypothesis that $\beta = 0.98$

In [12]:
T_test = res.t_test("x2=0.98")
print(T_test)
print(T_test.tvalue)
print(T_test.pvalue)

                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.9979      0.004      4.053      0.000       0.989       1.006
[[4.0529986]]
5.0565305457432177e-05


## Problem 4

**Problem 4a**

Read the dataset and perform the required manipulations

In [13]:
minwage = pd.read_csv('minwage.txt', sep="\t", na_values='.')

# Drop rows with missing values in considering all columns except 'chain', 'own', and 'state'
print(len(minwage))
rel_cols = [col for col in minwage.columns if col not in ['chain', 'own', 'state']]
minwage = minwage.dropna(subset=rel_cols).reset_index(drop=True)
minwage = minwage.astype({col: 'float' for col in rel_cols}) # Fix data type
print(len(minwage))

# Add columns for the full-time equivalent variables
for suffix in ['', '2']:
    minwage[f'fte{suffix}'] = minwage[f'empft{suffix}'] + 0.5 * minwage[f'emppt{suffix}'] + minwage[f'nmgrs{suffix}']

minwage.head()

410
351


,chain,own,state,empft,emppt,nmgrs,wagest,empft2,emppt2,nmgrs2,wagest2,fte,fte2
0,4,1,0,20.0,20.0,4.0,5.00,0.0,36.0,2.0,5.25,34.0,20.0
1,4,1,0,6.0,26.0,5.0,5.50,28.0,3.0,6.0,4.75,24.0,35.5
2,1,0,0,50.0,35.0,3.0,5.00,15.0,18.0,5.0,4.75,70.5,29.0
3,1,0,0,10.0,17.0,5.0,5.00,26.0,9.0,6.0,5.00,23.5,36.5
4,2,1,0,2.0,8.0,5.0,5.25,3.0,12.0,2.0,5.00,11.0,11.0


Compute the averages to fill in the box

In [14]:
avg_pa_before = minwage[minwage['state'] == 0]['fte'].mean()
avg_pa_after = minwage[minwage['state'] == 0]['fte2'].mean()
avg_nj_before = minwage[minwage['state'] == 1]['fte'].mean()
avg_nj_after = minwage[minwage['state'] == 1]['fte2'].mean()

# Store results in pandas DataFrame
table = pd.DataFrame(data={'PA': [avg_pa_before, avg_pa_after], 'NJ': [avg_nj_before, avg_nj_after]}, index=['Before', 'After'])

# Add column for the column-wise difference
table['Difference'] = table['NJ'] - table['PA']

# Add row for the row-wise difference   
table.loc['Difference'] = table.loc['After'] - table.loc['Before']

# Check that diff-in-diff is coincides if we look at rows and columns
print(
    table.loc['Difference', 'Difference'] == (table.loc['Difference', 'NJ'] - table.loc['Difference', 'PA']) == (table.loc['After', 'Difference'] - table.loc['Before', 'Difference'])
)

display(table)

# Export table to LaTeX
latex_str = r'''
\begin{table}[!htbp] 
\caption{Average FTE before and after NJ minimum wage increase}
\label{tab:problem4a}
\begin{center}
''' + '\n'+ table.to_latex(float_format="{:.3f}".format) + '\n' + r'''
\end{center}
\end{table}
'''
with open('p4a_table.tex', 'w') as f:
    f.write(latex_str)

True


,PA,NJ,Difference
Before,23.704545,20.678246,-3.026300
After,21.825758,21.076316,-0.749442
Difference,-1.878788,0.398070,2.276858


**Problem 4b**

Re-shape the dataset

In [15]:
minwage_melt = minwage.melt(id_vars=['chain', 'own', 'state', 'wagest'], value_vars=['fte', 'fte2'], var_name='time', value_name='fte_value')
minwage_melt['time'] = minwage_melt['time'].map({'fte': 0, 'fte2': 1})
minwage_melt.head()

,chain,own,state,wagest,time,fte_value
0,4,1,0,5.00,0,34.0
1,4,1,0,5.50,0,24.0
2,1,0,0,5.00,0,70.5
3,1,0,0,5.00,0,23.5
4,2,1,0,5.25,0,11.0


Compute the Diffs-in-Diffs estimate from Problem 4a via OLS regression

In [16]:
stata.pdataframe_to_data(minwage_melt[['state', 'time', 'fte_value']], force=True)

stata_code = f'''
capture ssc install outreg2

label var state "State (0=PA, 1=NJ)"
label var time "Time (0=Before, 1=After)"

* Regress fte_value on state, time, and their interaction
reg fte_value i.state##i.time
outreg2 using "p4b_table.tex", replace tex(fragment) ctitle("FTE") dec(4) keep(1.state 1.time 1.state#1.time) label 
'''

stata.run(stata_code)


. 
. capture ssc install outreg2

. 
. label var state "State (0=PA, 1=NJ)"

. label var time "Time (0=Before, 1=After)"

. 
. * Regress fte_value on state, time, and their interaction
. reg fte_value i.state##i.time

      Source |       SS           df       MS      Number of obs   =       702
-------------+----------------------------------   F(3, 698)       =      2.00
       Model |  521.059096         3  173.686365   Prob > F        =    0.1134
    Residual |  60761.1226       698  87.0503189   R-squared       =    0.0085
-------------+----------------------------------   Adj R-squared   =    0.0042
       Total |  61282.1817       701  87.4210866   Root MSE        =    9.3301

------------------------------------------------------------------------------
   fte_value | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
     1.state |    -3.0263   1.274513    -2.37   0.018    -5.528639   

Fix LaTeX output

In [17]:
with open('p4b_table.tex', 'r') as f:
    latex_str = f.read()
replacements = {
    'VARIABLES': '',
    'State (0=PA, 1=NJ) = 1': r'$\mathbb{1}$(NJ)', 
    'Time (0=Before, 1=After) = 1': r'$\mathbb{1}$(After)',
    r'1.state\#1.time': r'$\mathbb{1}$(NJ) $\times$ $\mathbb{1}$(After)',
}
for old, new in replacements.items(): latex_str = latex_str.replace(old, new)
latex_str = r'''
\begin{table}[!htbp] 
\caption{Differences-in-differences regression for FTE}
\label{tab:problem4b}
\begin{center}
''' + '\n'+ latex_str + '\n' + r'''
\end{center}
\end{table}
'''
with open('p4b_table.tex', 'w') as f:
    f.write(latex_str)

**Problem 4c**

In [18]:
stata.pdataframe_to_data(minwage_melt, force=True)

stata_code = f'''
capture ssc install outreg2

label var state "State (0=PA, 1=NJ)"
label var time "Time (0=Before, 1=After)"
label var chain "Restaurant Chain"
label var own "Ownership Status (1=Company-owned, 0=Otherwise)"

label define chainlabel 1 "Burger King" 2 "KFC" 3 "Wendy's" 4 "Roy Rogers"
label values chain chainlabel

* Regress fte_value on state, time, and their interaction, controlling for chain and ownership
reg fte_value i.state##i.time i.chain i.own
outreg2 using "p4c_table.tex", replace tex(fragment) ctitle("FTE") dec(4) keep(1.state 1.time 1.state#1.time i.chain 1.own) label 
'''

stata.run(stata_code)


. 
. capture ssc install outreg2

. 
. label var state "State (0=PA, 1=NJ)"

. label var time "Time (0=Before, 1=After)"

. label var chain "Restaurant Chain"

. label var own "Ownership Status (1=Company-owned, 0=Otherwise)"

. 
. label define chainlabel 1 "Burger King" 2 "KFC" 3 "Wendy's" 4 "Roy Rogers"

. label values chain chainlabel

. 
. * Regress fte_value on state, time, and their interaction, controlling for ch
> ain and ownership
. reg fte_value i.state##i.time i.chain i.own

      Source |       SS           df       MS      Number of obs   =       702
-------------+----------------------------------   F(7, 694)       =     27.70
       Model |  13382.0568         7   1911.7224   Prob > F        =    0.0000
    Residual |  47900.1249       694  69.0203528   R-squared       =    0.2184
-------------+----------------------------------   Adj R-squared   =    0.2105
       Total |  61282.1817       701  87.4210866   Root MSE        =    8.3078

---------------------------------

Fix LaTeX output

In [19]:
with open('p4c_table.tex', 'r') as f:
    latex_str = f.read()
replacements.update(
    {"Restaurant Chain = 2, KFC": r"$\mathbb{1}$(KFC)",
     "Restaurant Chain = 3, Wendy's": r"$\mathbb{1}$(Wendy's)",
     "Restaurant Chain = 4, Roy Rogers": r"$\mathbb{1}$(Roy Rogers)",
     "Ownership Status (1=Company-owned, 0=Otherwise) = 1": r"$\mathbb{1}$(Company-owned)",   
    }
)
for old, new in replacements.items(): latex_str = latex_str.replace(old, new)
latex_str = r'''
\begin{table}[!htbp] 
\caption{Differences-in-differences regression for FTE with controls for chain and ownership}
\label{tab:problem4c}
\begin{center}
''' + '\n'+ latex_str + '\n' + r'''
\end{center}
\end{table}
'''
with open('p4c_table.tex', 'w') as f:
    f.write(latex_str)

**Problem 2d**

Considering minwage (not the reshaped version!), add variable for the change in full-time employment

In [20]:
minwage['fte_change'] = minwage['fte2'] - minwage['fte']
minwage.head()

,chain,own,state,empft,emppt,nmgrs,wagest,empft2,emppt2,nmgrs2,wagest2,fte,fte2,fte_change
0,4,1,0,20.0,20.0,4.0,5.00,0.0,36.0,2.0,5.25,34.0,20.0,-14.0
1,4,1,0,6.0,26.0,5.0,5.50,28.0,3.0,6.0,4.75,24.0,35.5,11.5
2,1,0,0,50.0,35.0,3.0,5.00,15.0,18.0,5.0,4.75,70.5,29.0,-41.5
3,1,0,0,10.0,17.0,5.0,5.00,26.0,9.0,6.0,5.00,23.5,36.5,13.0
4,2,1,0,2.0,8.0,5.0,5.25,3.0,12.0,2.0,5.00,11.0,11.0,0.0


Construct the gap variable following Card and Krueger

Note: Given the initial wage $W_{1i}$,

$$GAP_{i}=\begin{cases}
0 & \text{for stores in Pennsylvania or stores in New Jersey with }W_{1i} \ge 5.05 \\
\frac{5.05-W_{1i}}{W_{1i}} & \text{for other stores in New Jersey}
\end{cases}$$

In [21]:
minwage['gap'] = np.zeros(len(minwage))
cond = (minwage['state'] == 1) & (minwage['wagest'] < 5.05)
minwage.loc[cond, 'gap'] = (5.05 - minwage.loc[cond, 'wagest']) / minwage.loc[cond, 'wagest']
minwage.head()

,chain,own,state,empft,emppt,nmgrs,wagest,empft2,emppt2,nmgrs2,wagest2,fte,fte2,fte_change,gap
0,4,1,0,20.0,20.0,4.0,5.00,0.0,36.0,2.0,5.25,34.0,20.0,-14.0,0.0
1,4,1,0,6.0,26.0,5.0,5.50,28.0,3.0,6.0,4.75,24.0,35.5,11.5,0.0
2,1,0,0,50.0,35.0,3.0,5.00,15.0,18.0,5.0,4.75,70.5,29.0,-41.5,0.0
3,1,0,0,10.0,17.0,5.0,5.00,26.0,9.0,6.0,5.00,23.5,36.5,13.0,0.0
4,2,1,0,2.0,8.0,5.0,5.25,3.0,12.0,2.0,5.00,11.0,11.0,0.0,0.0


Regress the change in full-time employment on gap

In [22]:
stata.pdataframe_to_data(minwage, force=True)

stata_code = f'''
capture ssc install outreg2

label var fte_change "Change in FTE"
label var gap "Required wage increase (%)"


* Regress fte_change on gap
reg fte_change gap
outreg2 using "p4d_table.tex", replace tex(fragment) ctitle("Change in FTE") dec(4) label 
'''

stata.run(stata_code)


. 
. capture ssc install outreg2

. 
. label var fte_change "Change in FTE"

. label var gap "Required wage increase (%)"

. 
. 
. * Regress fte_change on gap
. reg fte_change gap

      Source |       SS           df       MS      Number of obs   =       351
-------------+----------------------------------   F(1, 349)       =      7.84
       Model |  588.370886         1  588.370886   Prob > F        =    0.0054
    Residual |  26193.9395       349  75.0542679   R-squared       =    0.0220
-------------+----------------------------------   Adj R-squared   =    0.0192
       Total |  26782.3104       350  76.5208869   Root MSE        =    8.6634

------------------------------------------------------------------------------
  fte_change | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         gap |   17.05158   6.090131     2.80   0.005     5.073601    29.02955
       _cons |  -1.477734  

Fix LaTeX output

In [23]:
with open('p4d_table.tex', 'r') as f:
    latex_str = f.read()
for old, new in replacements.items(): latex_str = latex_str.replace(old, new)
latex_str = r'''
\begin{table}[!htbp] 
\caption{Regression of the change in FTE on the required proportional wage increase}
\label{tab:problem4d}
\begin{center}
''' + '\n'+ latex_str + '\n' + r'''
\end{center}
\end{table}
'''
with open('p4d_table.tex', 'w') as f:
    f.write(latex_str)

Store the coefficient estimate on gap in python variable

In [24]:
beta_gap = stata.get_ereturn()['e(b)'][0][0]
print(beta_gap)

17.051576930218513


Multiply the coefficient estimate on gap by the mean gap for NJ restaurants

In [25]:
print(beta_gap * minwage[minwage['state'] == 1]['gap'].mean())

1.7829285215831516


**Problem 2e**

Run same regression as in Problem 2d but add the state dummy variable and control for restaurant chain and ownership

In [ ]:
stata.pdataframe_to_data(minwage, force=True)

stata_code = f'''
capture ssc install outreg2

label var fte_change "Change in FTE"
label var gap "Required wage increase (%)"
label var state "State (0=PA, 1=NJ)"
label var chain "Restaurant Chain"
label var own "Ownership Status (1=Company-owned, 0=Otherwise)"

label define chainlabel 1 "Burger King" 2 "KFC" 3 "Wendy's" 4 "Roy Rogers"
label values chain chainlabel

* Regress fte_change on gap, state, and controls for chain and ownership
reg fte_change gap i.state i.chain i.own
outreg2 using "p4e_table.tex", replace tex(fragment) ctitle("Change in FTE") dec(4) label 
'''

stata.run(stata_code)


. 
. capture ssc install outreg2

. 
. label var fte_change "Change in FTE"

. label var gap "Required wage increase (%)"

. label var state "State (0=PA, 1=NJ)"

. label var chain "Restaurant Chain"

. label var own "Ownership Status (1=Company-owned, 0=Otherwise)"

. 
. label define chainlabel 1 "Burger King" 2 "KFC" 3 "Wendy's" 4 "Roy Rogers"

. label values chain chainlabel

. 
. * Regress fte_value on gap, state, and controls for chain and ownership
. reg fte_change gap i.state i.chain i.own

      Source |       SS           df       MS      Number of obs   =       351
-------------+----------------------------------   F(6, 344)       =      1.80
       Model |  813.949111         6  135.658185   Prob > F        =    0.0988
    Residual |  25968.3613       344  75.4894223   R-squared       =    0.0304
-------------+----------------------------------   Adj R-squared   =    0.0135
       Total |  26782.3104       350  76.5208869   Root MSE        =    8.6885

---------------------

Fix LaTeX output

In [27]:
with open('p4e_table.tex', 'r') as f:
    latex_str = f.read()
for old, new in replacements.items(): latex_str = latex_str.replace(old, new)
latex_str = r'''
\begin{table}[!htbp] 
\caption{Regression of the change in FTE on the required proportional wage increase, controlling for state, chain, and ownership}
\label{tab:problem4e}
\begin{center}
''' + '\n'+ latex_str + '\n' + r'''
\end{center}
\end{table}
'''
with open('p4e_table.tex', 'w') as f:
    f.write(latex_str)